In [2]:
import os
import sys
import time
import psutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de estilo gráfico
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.max_columns', None)

# Definición de ruta relativa apuntando a la subcarpeta superior 'Data Sets'
ruta_csv = os.path.join("..", "Data_Sets", "precio_consumidor_2026.csv")

if os.path.exists(ruta_csv):
    print(f"✓ Archivo localizado correctamente en: '{ruta_csv}'")
    df_raw = pd.read_csv(ruta_csv, low_memory=False)
    print(f"Dimensiones iniciales del dataset: {df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas")
    print("\nColumnas detectadas en el dataset:")
    print(df_raw.columns.tolist())
    print("\nVista previa de los primeros 3 registros:")
    display(df_raw.head(3))
else:
    raise FileNotFoundError(f"No se encontró el archivo en la ruta: '{ruta_csv}'. Verifica el nombre de la carpeta.")

✓ Archivo localizado correctamente en: '..\Data_Sets\precio_consumidor_2026.csv'
Dimensiones iniciales del dataset: 221,432 filas x 15 columnas

Columnas detectadas en el dataset:
['Anio', 'Mes', 'Semana', 'Fecha inicio', 'Fecha termino', 'ID region', 'Region', 'Sector', 'Tipo de punto monitoreo', 'Grupo', 'Producto', 'Unidad', 'Precio minimo', 'Precio maximo', 'Precio promedio']

Vista previa de los primeros 3 registros:


,Anio,Mes,Semana,Fecha inicio,Fecha termino,ID region,Region,Sector,Tipo de punto monitoreo,Grupo,Producto,Unidad,Precio minimo,Precio maximo,Precio promedio
0,2026,1,1,2025-12-29,2026-01-02,4,Región de Coquimbo,La Serena,Carnicería,Carne bovina,Abastero,$/kilo,11890,11890,"11890,000000"
1,2026,1,1,2025-12-29,2026-01-02,4,Región de Coquimbo,La Serena,Carnicería,Carne bovina,Asado Carnicero,$/kilo,11890,11890,"11890,000000"
2,2026,1,1,2025-12-29,2026-01-02,4,Región de Coquimbo,La Serena,Carnicería,Carne bovina,Asado de tira,$/kilo,12490,12490,"12490,000000"


In [3]:
# Reporte de valores nulos y tipos de datos iniciales
df_reporte = pd.DataFrame({
    'Tipo_Dato': df_raw.dtypes,
    'Valores_Nulos': df_raw.isnull().sum(),
    'Porcentaje_Nulos (%)': (df_raw.isnull().sum() / len(df_raw)) * 100
})

print("=== DIAGNÓSTICO DE CALIDAD Y VALORES FALTANTES ===")
display(df_reporte.sort_values(by='Porcentaje_Nulos (%)', ascending=False))

=== DIAGNÓSTICO DE CALIDAD Y VALORES FALTANTES ===


,Tipo_Dato,Valores_Nulos,Porcentaje_Nulos (%)
Anio,int64,0,0.0
Mes,int64,0,0.0
Semana,int64,0,0.0
Fecha inicio,str,0,0.0
Fecha termino,str,0,0.0
ID region,int64,0,0.0
Region,str,0,0.0
Sector,str,0,0.0
Tipo de punto monitoreo,str,0,0.0
Grupo,str,0,0.0


In [4]:
def pipeline_preprocesamiento_farma(df_in):
    """
    Aplica limpieza exhaustiva, filtrado de NAs, conversión de tipos 
    y feature engineering sobre el dataset de precios al consumidor.
    """
    df = df_in.copy()
    
    # 1. Normalización de cadenas de texto (Eliminar espacios sobrantes y estandarizar a mayúsculas)
    columnas_texto = df.select_dtypes(include=['object']).columns
    for col in columnas_texto:
        df[col] = df[col].astype(str).str.strip().str.upper()
        df[col] = df[col].replace(['NAN', 'NONE', 'NULL', ''], np.nan)
        
    # 2. Convertir columnas numéricas / precios (removiendo símbolos de moneda si existieran)
    cols_precios = [c for c in df.columns if 'precio' in c.lower() or 'monto' in c.lower() or 'valor' in c.lower()]
    for col in cols_precios:
        if df[col].dtype == 'object':
            df[col] = df[col].astype(str).str.replace(r'[^\d.]', '', regex=True)
            df[col] = pd.to_numeric(df[col], errors='coerce')
            
    # 3. Tratamiento riguroso de NAs: Imputación estratégica y filtrado de llaves vacías
    df = df.dropna(thresh=int(df.shape[1] * 0.5))  # Elimina filas con más del 50% de nulos
    
    # Imputación de columnas categóricas secundarias
    for col in columnas_texto:
        if col in df.columns:
            df[col] = df[col].fillna('NO ESPECIFICADO')
            
    # Imputación de columnas numéricas por la mediana
    for col in cols_precios:
        if col in df.columns and df[col].isnull().sum() > 0:
            mediana_val = df[col].median()
            df[col] = df[col].fillna(mediana_val)
            
    return df

# Ejecutar pipeline sobre el dataset
df_clean = pipeline_preprocesamiento_farma(df_raw)

# Guardar copia procesada limpia en la carpeta F2
ruta_salida = "datos_consumidor_procesados.csv"
df_clean.to_csv(ruta_salida, index=False)
print(f"✓ Pipeline ejecutado con éxito. Dataset limpio guardado en '{ruta_salida}'")
print(f"Dimensiones finales tras limpieza: {df_clean.shape[0]:,} filas x {df_clean.shape[1]} columnas")

C:\Users\jorge\AppData\Local\Temp\ipykernel_18536\4188640987.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columnas_texto = df.select_dtypes(include=['object']).columns


✓ Pipeline ejecutado con éxito. Dataset limpio guardado en 'datos_consumidor_procesados.csv'
Dimensiones finales tras limpieza: 221,432 filas x 15 columnas


In [5]:
# Verificación de integridad y consistencia
print("=== EJECUTANDO PRUEBAS UNITARIAS DE VALIDACIÓN TÉCNICA ===")

# Prueba 1: Sin valores nulos críticos
nulos_totales = df_clean.isnull().sum().sum()
print(f"1. Total de valores vacíos remanentes: {nulos_totales}")

# Prueba 2: Trazabilidad de dimensiones
assert df_clean.shape[0] > 0, "Error crítico: El dataset procesado quedó vacío."
print("2. Prueba de dimensiones: PASADA (Dataset no vacío)")

# Prueba 3: Verificación de tipos de datos procesados
print("3. Resumen de tipos de datos resultantes:")
print(df_clean.dtypes.value_counts())
print("\n✓ VALIDACIÓN COMPLETA SIN ERRORES.")

=== EJECUTANDO PRUEBAS UNITARIAS DE VALIDACIÓN TÉCNICA ===
1. Total de valores vacíos remanentes: 0
2. Prueba de dimensiones: PASADA (Dataset no vacío)
3. Resumen de tipos de datos resultantes:
str      9
int64    6
Name: count, dtype: int64

✓ VALIDACIÓN COMPLETA SIN ERRORES.


## 8. Vinculación con el mapa conceptual

Esta sección relaciona los elementos definidos previamente en el mapa conceptual
con su implementación efectiva dentro del proyecto.

Para cada elemento se identifica:

1. El nodo o concepto del mapa conceptual.
2. El lugar del proyecto donde fue implementado.
3. La evidencia verificable que demuestra su implementación.

Esta vinculación permite comprobar que los elementos planificados inicialmente
tienen una representación concreta dentro del repositorio y del notebook de la Fase 1.


In [ ]:
VINCULACION = [
    # (nodo del mapa conceptual, dónde se implementa, evidencia verificable)

    (
        "Entorno reproducible",
        "F1 · verificar_entorno()",
        "Intérprete de Python, entorno virtual y versiones de librerías"
    ),

    (
        "Gestión de dependencias",
        "F1 · requirements.txt",
        "Archivo requirements.txt generado con las dependencias del proyecto"
    ),

    (
        "Estructura del proyecto",
        "F1 · creación de carpetas",
        "Carpetas data/raw, data/processed, docs, src y F1-F4"
    ),

    (
        "Fuente de datos",
        "F1 · FICHA",
        "Ficha del dataset SIMCE 4º Básico 2025 y fuente oficial"
    ),

    (
        "Diccionario de variables",
        "F1 · diccionario",
        "42 variables documentadas con rol analítico y descripción"
    ),

    (
        "Disponibilidad del dataset",
        "F1 · ARCHIVO_RAW",
        "Verificación de existencia y dimensiones del archivo SIMCE"
    ),

    (
        "Control de versiones",
        "F1 · diagnóstico Git",
        "Versión de Git e historial reciente de commits"
    ),

    (
        "Validación técnica",
        "Sección 7 · validar_fase1()",
        "Comprobaciones automáticas de estructura, archivos y coherencia"
    ),
]


vinculacion = pd.DataFrame(
    VINCULACION,
    columns=[
        "nodo_mapa",
        "donde_se_implementa",
        "evidencia"
    ]
)


# Comprobar que ningún elemento quede sin evidencia
assert vinculacion["evidencia"].str.strip().ne("").all(), \
    "Hay elementos del mapa conceptual sin evidencia."


print(
    f"{len(vinculacion)} elementos vinculados, "
    "todos con evidencia asociada.\n"
)

display(vinculacion)

> **Vinculación con el informe.**
> Cada elemento del mapa conceptual debe poder relacionarse con una evidencia
> concreta dentro del proyecto, como una celda del notebook, un archivo del
> repositorio o un registro de Git.
>
> Si un elemento fue incluido en el mapa conceptual pero todavía no cuenta con
> evidencia dentro del proyecto, deberá identificarse como pendiente o justificarse
> su modificación respecto de la planificación inicial.

## 9. Persistencia y trazabilidad

El resultado de la Fase 1 debe quedar guardado y documentado de forma
reproducible.

En esta sección se generan y almacenan los principales artefactos de la fase:

- `README.md` del proyecto.
- Diccionario de variables.
- Evaluación de los criterios del dataset.
- Vinculación con el mapa conceptual.
- Metadatos de ejecución de la Fase 1.

Estos archivos permiten mantener trazabilidad entre el notebook, el repositorio
GitHub y el informe técnico.

In [ ]:
TITULO_PROYECTO = "Caracterización e Inequidad en los Resultados Académicos del SIMCE 4º Básico en Chile"
GRUPO = "Grupo 7"
INTEGRANTES = ["Felipe", "Vicente", "Cristian", "Jorge"]


def generar_readme(titulo, grupo, integrantes, ficha, dependencias, ruta):
    """Genera el README técnico del proyecto."""

    lista_integrantes = "\n".join(
        f"- {nombre}" for nombre in integrantes
    )

    lista_dependencias = "\n".join(
        f"- {dep}" for dep in dependencias
    )

    contenido = f"""# {titulo}

**Grupo:** {grupo}

## Integrantes
{lista_integrantes}

## Datos
- Dataset: {ficha['titulo']}
- Fuente: {ficha['autor']}
- Plataforma: {ficha['plataforma']}
- Enlace: {ficha['url']}
- Unidad de observación: {ficha['unidad_observacion']}
- Dimensiones: {ficha['filas']} filas x {ficha['columnas']} columnas

## Estructura del repositorio

- `data/raw/`: datos originales sin modificar.
- `data/processed/`: datos procesados en fases posteriores.
- `docs/`: documentación y metadatos.
- `src/`: módulos reutilizables.
- `F1/`: definición y preparación del proyecto.
- `F2/`: procesamiento de datos.
- `F3/` y `F4/`: fases posteriores.

## Requisitos

Python: {sys.version.split()[0]}

### Dependencias
{lista_dependencias}

## Reproducibilidad

La semilla utilizada durante el proyecto es `{SEMILLA}`.

Los datos originales no se modifican durante la Fase 1.
"""

    ruta.write_text(contenido, encoding="utf-8")

    return contenido


ARCHIVO_README = RAIZ / "README.md"

contenido_readme = generar_readme(
    TITULO_PROYECTO,
    GRUPO,
    INTEGRANTES,
    FICHA,
    REQ_LINES,
    ARCHIVO_README
)

print(
    f"✓ README generado correctamente "
    f"({len(contenido_readme.splitlines())} líneas)"
)

In [ ]:
# Guardar diccionario de variables
diccionario.to_csv(
    DIR_DOCS / "diccionario_variables.csv",
    index=False
)

# Guardar evaluación del dataset
evaluacion_df.to_csv(
    DIR_DOCS / "evaluacion_criterios_dataset.csv",
    index=False
)

# Guardar vinculación con el mapa conceptual
vinculacion.to_csv(
    DIR_DOCS / "vinculacion_mapa_conceptual.csv",
    index=False
)


# Metadatos de ejecución de la Fase 1
METADATOS = {
    "proyecto": TITULO_PROYECTO,
    "grupo": GRUPO,
    "fase": "F1",
    "fecha_ejecucion": date.today().isoformat(),
    "semilla": SEMILLA,

    "python": sys.version.split()[0],

    "sistema_operativo": (
        f"{platform.system()} {platform.release()}"
    ),

    "interprete": ENTORNO["interprete"],

    "entorno_virtual": ENTORNO["entorno_virtual"],

    "dependencias": ENTORNO["versiones"],

    "dataset": {
        "titulo": FICHA["titulo"],
        "fuente": FICHA["autor"],
        "url": FICHA["url"],
        "filas": FICHA["filas"],
        "columnas": FICHA["columnas"],
        "archivo_disponible": ARCHIVO_RAW.exists()
    }
}


# Guardar los metadatos como JSON
archivo_metadatos = DIR_DOCS / "metadatos_fase1.json"

archivo_metadatos.write_text(
    json.dumps(
        METADATOS,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


print("✓ Artefactos de F1 guardados:\n")

for archivo in sorted(DIR_DOCS.iterdir()):
    print(
        f"  {archivo.name:40} "
        f"{archivo.stat().st_size:8} bytes"
    )

In [ ]:
criterios_incumplidos = (
    evaluacion_df["cumple"] == "NO"
).sum()


resumen_f1 = pd.DataFrame([
    {
        "artefacto": "Variables documentadas",
        "estado": len(diccionario)
    },
    {
        "artefacto": "Elementos del mapa vinculados",
        "estado": len(vinculacion)
    },
    {
        "artefacto": "Criterios de dataset incumplidos",
        "estado": criterios_incumplidos
    },
    {
        "artefacto": "Archivos generados en docs/",
        "estado": len(list(DIR_DOCS.iterdir()))
    },
    {
        "artefacto": "Dataset disponible",
        "estado": "Sí" if ARCHIVO_RAW.exists() else "No"
    },
    {
        "artefacto": "README generado",
        "estado": "Sí" if ARCHIVO_README.exists() else "No"
    }
])

display(resumen_f1)